# 面试题：Context 怎样选择并遵守 token 预算？

面试回答：context 是带预算的证据选择；强制保留策略、权限、目标、权威状态和 schema，再按相关性、时效、风险与 token 成本选择可选片段。这是行动前的证据门禁，不是普通摘要压缩。

## 真实案例

退款助手有 80 token 预算，需要在政策、订单、schema、聊天、旧摘要和商品文案中选择。

## 基线

基线只按最近到达顺序截断。

## 结果解读

输出每段成本、价值与是否强制。

## 失败案例

最近文本会挤掉退款 schema，造成后续调用不安全。

In [1]:
chunks = [{'kind':'policy','tokens':18,'value':10,'required':True}, {'kind':'order','tokens':22,'value':9,'required':True}, {'kind':'schema','tokens':16,'value':9,'required':True}, {'kind':'chat','tokens':20,'value':4,'required':False}, {'kind':'old_summary','tokens':14,'value':2,'required':False}, {'kind':'copy','tokens':17,'value':1,'required':False}]  # 构造六段带 token 成本、价值和强制语义的候选证据。
budget = 65  # 设置当前退款动作的 token 硬预算。
print('候选:', chunks)  # 输出所有 context 候选。
print('预算:', budget)  # 输出固定预算。

候选: [{'kind': 'policy', 'tokens': 18, 'value': 10, 'required': True}, {'kind': 'order', 'tokens': 22, 'value': 9, 'required': True}, {'kind': 'schema', 'tokens': 16, 'value': 9, 'required': True}, {'kind': 'chat', 'tokens': 20, 'value': 4, 'required': False}, {'kind': 'old_summary', 'tokens': 14, 'value': 2, 'required': False}, {'kind': 'copy', 'tokens': 17, 'value': 1, 'required': False}]
预算: 65


In [2]:
used = 0  # 初始化最近截断基线的 token 使用量。
recent_keep = []  # 初始化最近截断基线的保留集合。
for item in reversed(chunks):  # 从最晚到达的片段开始塞入上下文。
    if used + item['tokens'] <= budget:  # 仅依据 token 容量判断。
        recent_keep.append(item['kind'])  # 记录没有质量门槛的保留片段。
        used += item['tokens']  # 更新基线占用。
print('最近截断:', recent_keep, used)  # 输出可能缺少强制 schema 的基线。

最近截断: ['copy', 'old_summary', 'chat'] 51


In [3]:
def choose(items, limit):  # 定义先保强制证据再选择可选证据的手写选择器。
    picked = [item for item in items if item['required']]  # 保留不能被截断的政策、状态和契约。
    spent = sum(item['tokens'] for item in picked)  # 计算强制证据占用。
    for item in sorted((item for item in items if not item['required']), key=lambda row:row['value'] / row['tokens'], reverse=True):  # 按单位 token 价值处理可选项。
        if spent + item['tokens'] <= limit:  # 检查加入当前片段是否仍符合预算。
            picked.append(item)  # 保存具备性价比的可选证据。
            spent += item['tokens']  # 更新选择器的实际 token 使用量。
    return picked, spent  # 返回最终证据包和成本。

In [4]:
picked, spent = choose(chunks, budget)  # 在相同预算下执行安全 context 选择。
print('kind | tokens | value | required')  # 输出结果表标题。
for item in picked:  # 逐条查看实际进入 prompt 的证据。
    print(item['kind'], item['tokens'], item['value'], item['required'])  # 输出选择依据。
print('总成本:', spent, '，schema 保留:', 'schema' in [item['kind'] for item in picked])  # 输出预算与关键契约是否保留。

kind | tokens | value | required
policy 18 10 True
order 22 9 True
schema 16 9 True
总成本: 56 ，schema 保留: True


In [5]:
wrong = 'schema' in recent_keep  # 检查最近截断是否错误地保留关键 schema。
fixed = 'schema' in [item['kind'] for item in picked]  # 检查受约束选择是否保留 schema。
print('失败案例：最近截断 schema=', wrong, '，安全选择=', fixed)  # 展示简单截断的安全风险。
print('生产差距：需接真实 tokenizer、来源时间、相关性模型、ACL 与安全余量。')  # 说明生产选择器要求。

失败案例：最近截断 schema= False ，安全选择= True
生产差距：需接真实 tokenizer、来源时间、相关性模型、ACL 与安全余量。


In [6]:
assert spent <= budget  # 验证选择器不会超出 token 预算。
assert fixed  # 验证工具 schema 始终保留。
assert not wrong  # 验证最近截断反例确实丢失关键 schema。